In [ ]:
import sys
sys.path.append('..')
import pandas as pd
from utils.db_utils import write_table, read_table
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, r2_score


In [85]:
df = read_table("""
    SELECT *
    FROM sc_silver.gdp_per_capita
    """)

In [86]:
df

,state,date,gdp_rm_million,gdp_growth_yoy,population,gdp_per_capita
0,Johor,2016-01-01,114965.455,6.023904,3651800.0,31481.859631
1,Johor,2017-01-01,121697.116,5.978778,3697000.0,32917.802543
2,Johor,2018-01-01,128914.397,5.988315,3749400.0,34382.673761
3,Johor,2019-01-01,132617.111,3.589793,3761200.0,35259.255291
4,Johor,2020-01-01,126746.445,-3.751183,4009700.0,31609.957104
...,...,...,...,...,...,...
139,W.P. Putrajaya,2020-01-01,40697.536,-11.021000,109200.0,372688.058608
140,W.P. Putrajaya,2021-01-01,41895.612,2.944000,115200.0,363677.187500
141,W.P. Putrajaya,2022-01-01,42872.705,2.332000,117000.0,366433.376068
142,W.P. Putrajaya,2023-01-01,44503.210,3.803000,118800.0,374606.144781


In [87]:
forecast_years = range(2025, 2031)  # 2025 to 2030

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["state", "date"]).reset_index(drop=True)

results = []
metrics = []

for state, sdf in df.groupby("state"):
    sdf = sdf.sort_values("date")
    y = sdf["gdp_rm_million"].values
    
    # --- Fit models ---
    try:
        hw_model = ExponentialSmoothing(y, trend="add", damped_trend=True).fit()
        hw_forecast = hw_model.forecast(len(forecast_years))
    except:
        hw_forecast = np.full(len(forecast_years), np.nan)
    
    try:
        arima_model = auto_arima(y, seasonal=False, suppress_warnings=True)
        arima_forecast = arima_model.predict(n_periods=len(forecast_years))
    except:
        arima_forecast = np.full(len(forecast_years), np.nan)
    
    # --- Choose best model by MAE on last 3 points ---
    def mae(y_true, y_pred):
        return np.mean(np.abs(y_true[-3:] - y_pred[-3:]))
    
    hw_mae = mae(y, np.concatenate([y, hw_forecast])) if len(y) > 3 else np.inf
    arima_mae = mae(y, np.concatenate([y, arima_forecast])) if len(y) > 3 else np.inf
    
    if hw_mae <= arima_mae:
        forecast = hw_forecast
        model_type = "Holt-Winters"
    else:
        forecast = arima_forecast
        model_type = "ARIMA"
    
    # --- Store metrics ---
    try:
        fitted = hw_model.fittedvalues if model_type=="Holt-Winters" else arima_model.predict_in_sample()
        metrics.append({
            "state": state,
            "model": model_type,
            "train_r2": round(r2_score(y, fitted),4),
            "rmse": round(np.sqrt(mean_squared_error(y, fitted)),2)
        })
    except:
        metrics.append({"state": state, "model": model_type, "train_r2": None, "rmse": None})
    
    # --- Append forecast results, including last historical for growth ---
    last_hist_value = y[-1]
    full_forecast = np.insert(forecast, 0, last_hist_value)  # prepend last historical
    
    for i, year in enumerate(forecast_years):
        results.append({
            "state": state,
            "date": pd.Timestamp(f"{year}-01-01"),
            "gdp_rm_million": forecast[i],
            "legend": "forecast"
        })
    
# --- Historical dataframe ---
historical = df.copy()
historical["legend"] = "historical"

# --- Forecast dataframe ---
forecast_df = pd.DataFrame(results)
forecast_df = forecast_df.sort_values(["state", "date"]).reset_index(drop=True)

# Precompute last historical value per state for YoY base
last_hist_by_state = historical.groupby("state")["gdp_rm_million"].last()

# Forecast YoY should use last historical as the base for the first forecast

def calc_yoy(group):
    state = group.name
    last_hist = last_hist_by_state.loc[state]
    forecast_vals = group["gdp_rm_million"].values
    series = np.insert(forecast_vals, 0, last_hist)
    yoy = pd.Series(series).pct_change()[1:] * 100
    return yoy.values

forecast_df["gdp_growth_yoy"] = forecast_df.groupby("state").apply(calc_yoy).explode().values

# --- Combine historical and forecast ---
out_df = pd.concat([historical, forecast_df], ignore_index=True)
out_df = out_df.sort_values(["state", "date"]).reset_index(drop=True)
out_df = out_df[["state", "date", "gdp_growth_yoy", "legend"]]


In [88]:
out_df.head(20)

,state,date,gdp_growth_yoy,legend
0,Johor,2016-01-01,6.023904,historical
1,Johor,2017-01-01,5.978778,historical
2,Johor,2018-01-01,5.988315,historical
3,Johor,2019-01-01,3.589793,historical
4,Johor,2020-01-01,-3.751183,historical
5,Johor,2021-01-01,2.724879,historical
6,Johor,2022-01-01,8.463528,historical
7,Johor,2023-01-01,4.09425,historical
8,Johor,2024-01-01,6.786808,historical
9,Johor,2025-01-01,3.884131,forecast


In [89]:
df2 = read_table("""
    SELECT state, year, avg(unemp_rate) as unemp_rate
    FROM sc_bronze.dosm_graduates_state
    group by state, year
    order by state, year
    """)

df2

,state,year,unemp_rate
0,Johor,2016,5.05
1,Johor,2017,4.40
2,Johor,2018,3.30
3,Johor,2019,3.55
4,Johor,2020,4.50
...,...,...,...
139,W.P. Putrajaya,2020,1.45
140,W.P. Putrajaya,2021,1.50
141,W.P. Putrajaya,2022,1.95
142,W.P. Putrajaya,2023,1.10


In [90]:
forecast_years = range(2025, 2031)  # 2025 to 2030

# --- Convert year to date ---
unemp_data = df2.copy()
unemp_data["date"] = pd.to_datetime(unemp_data["year"].astype(str) + "-01-01")
unemp_data = unemp_data.sort_values(["state", "date"]).reset_index(drop=True)

forecast_results = []

for state, state_df in unemp_data.groupby("state"):
    state_df = state_df.sort_values("date")
    y = state_df["unemp_rate"].values

    # --- Fit models ---
    try:
        hw_model = ExponentialSmoothing(y, trend="add", damped_trend=True).fit()
        hw_forecast = hw_model.forecast(len(forecast_years))
    except:
        hw_forecast = np.full(len(forecast_years), np.nan)

    try:
        arima_model = auto_arima(y, seasonal=False, suppress_warnings=True)
        arima_forecast = arima_model.predict(n_periods=len(forecast_years))
    except:
        arima_forecast = np.full(len(forecast_years), np.nan)

    # --- Choose best model by MAE on last 3 points ---
    def mae(y_true, y_pred):
        return np.mean(np.abs(y_true[-3:] - y_pred[-3:]))

    hw_mae = mae(y, np.concatenate([y, hw_forecast])) if len(y) > 3 else np.inf
    arima_mae = mae(y, np.concatenate([y, arima_forecast])) if len(y) > 3 else np.inf

    if hw_mae <= arima_mae:
        forecast = hw_forecast
        model_type = "Holt-Winters"
    else:
        forecast = arima_forecast
        model_type = "ARIMA"

    # --- Append forecast results ---
    for i, year in enumerate(forecast_years):
        forecast_results.append({
            "state": state,
            "date": pd.Timestamp(f"{year}-01-01"),
            "unemp_rate": max(0, forecast[i]),  # clamp negative values
            "legend": "forecast"
        })

# --- Historical dataframe ---
historical_df = unemp_data.copy()
historical_df["legend"] = "historical"

# --- Forecast dataframe ---
forecast_df = pd.DataFrame(forecast_results)
forecast_df = forecast_df.sort_values(["state", "date"]).reset_index(drop=True)

# --- Combine historical and forecast ---
final_unemp_df = pd.concat([historical_df, forecast_df], ignore_index=True)
final_unemp_df = final_unemp_df.sort_values(["state", "date"]).reset_index(drop=True)

# --- Final columns ---
final_unemp_df = final_unemp_df[["state", "date", "unemp_rate","legend"]]
final_unemp_df

,state,date,unemp_rate,legend
0,Johor,2016-01-01,5.050000,historical
1,Johor,2017-01-01,4.400000,historical
2,Johor,2018-01-01,3.300000,historical
3,Johor,2019-01-01,3.550000,historical
4,Johor,2020-01-01,4.500000,historical
...,...,...,...,...
235,W.P. Putrajaya,2026-01-01,1.233460,forecast
236,W.P. Putrajaya,2027-01-01,1.223568,forecast
237,W.P. Putrajaya,2028-01-01,1.215654,forecast
238,W.P. Putrajaya,2029-01-01,1.209324,forecast


In [91]:
merged_df = pd.merge(
    out_df,
    final_unemp_df,
    on=["state", "date","legend"],
    how="inner" 
)
# Sort by state and date first
merged_df = merged_df.sort_values(["state", "date"]).reset_index(drop=True)

# Calculate delta_unemp = absolute change in unemployment rate
merged_df["delta_unemp"] = merged_df.groupby("state")["unemp_rate"].diff()

# Optional: drop rows with NaN (first year per state)
okun_df = merged_df.dropna(subset=["delta_unemp", "gdp_growth_yoy"])
okun_df

,state,date,gdp_growth_yoy,legend,unemp_rate,delta_unemp
1,Johor,2017-01-01,5.978778,historical,4.400000,-0.650000
2,Johor,2018-01-01,5.988315,historical,3.300000,-1.100000
3,Johor,2019-01-01,3.589793,historical,3.550000,0.250000
4,Johor,2020-01-01,-3.751183,historical,4.500000,0.950000
5,Johor,2021-01-01,2.724879,historical,4.100000,-0.400000
...,...,...,...,...,...,...
235,W.P. Putrajaya,2026-01-01,-0.132383,forecast,1.233460,-0.012365
236,W.P. Putrajaya,2027-01-01,-0.132383,forecast,1.223568,-0.009892
237,W.P. Putrajaya,2028-01-01,-0.132383,forecast,1.215654,-0.007913
238,W.P. Putrajaya,2029-01-01,-0.132383,forecast,1.209324,-0.006331


In [93]:
try:
    write_table(okun_df, 'sc_silver', 'forecast_gdp_unemp')
except Exception as e:
    print(f"Failed to write to Supabase: {e}")# 

Table sc_silver.forecast_gdp_unemp written successfully.
